In [1]:
# transferable code functions
import numpy as np

def padding(layer:np.ndarray, mode:str = 'zero'):


    padded_ly_size = (layer.shape[0]+2, layer.shape[1]+2)
    padded_ly = np.zeros(padded_ly_size)
    padded_ly[1:-1,1:-1] = layer
   
    if mode == 'continue':
        padded_ly[0,1:-1] = layer[0,:]
        padded_ly[-1,1:-1] = layer[-1,:]

        padded_ly[1:-1,0] = layer[:,0]
        padded_ly[1:-1,-1] = layer[:,-1]

        padded_ly[0,0] = layer[0,0]
        padded_ly[0,-1] = layer[0,-1]
        padded_ly[-1,0] = layer[-1,0]
        padded_ly[-1,-1] = layer[-1,-1]

    return(padded_ly)



In [2]:
import hydra
from omegaconf import DictConfig, OmegaConf
cfg = OmegaConf.load("model.yaml")


In [ ]:
#todo I'd like to create a version of this that can have dynamically sized convolutional layers
import numpy as np
from dataclasses import dataclass
from omegaconf import DictConfig

def build_index_lookup(cfg: DictConfig):
    """
    Given a loaded OmegaConf config with a 'layers' section,
    build a lookup table: index -> (layer_name, layer_data).
    """
    blocks = cfg.blocks

    index_lookup = {
        block_data.index: (block_name, block_data)
        for block_name, block_data in blocks.items()
    }
    return index_lookup

class Layer:
    #need to refactor this into a block that includes weights
    def __init__(self, ltype:str, index:int,  activations: np.ndarray, z_values: np.ndarray, dims:int, shape: tuple):
        self.activations = activations
        self.z_values = z_values
        self.ltype = ltype
        self.shape = shape
        self.dims = dims
        self.index = index

    @classmethod
    def from_shape(cls, layer_shape, l_type, index, dtype=np.float32, **kwargs):
        activations = np.zeros(layer_shape, dtype=dtype)
        z_vals = np.zeros(layer_shape, dtype=dtype)
        shape = activations.shape
        ltype = l_type
        dims = len(shape)
        index = index
        return cls(ltype, index, activations, z_vals, dims, shape, **kwargs)

class Input_block:
    def __init__(self, layer_shape, index, **kwargs): 
        #need to figure out how to properly use kwargs
        self.layer_shape = layer_shape
        self.index = index

        self.b_type = "input"
        self.ly_dim = len(layer_shape)
        self.activations = np.zeros(layer_shape)
        self.z_values = np.zeros(layer_shape)

class FC_block:
    def __init__(self, prev_ly_shape, layer_shape, index, **kwargs): 
        #need to figure out how to properly use kwargs
        self.layer_shape = layer_shape
        self.index = index
        self.prev_ly_shape = prev_ly_shape

        self.b_type = "fc"
        self.ly_dim = len(layer_shape)
        self.activations = np.zeros(layer_shape)
        self.z_values = np.zeros(layer_shape)
        self.biases = np.zeros(layer_shape)
        self.weights = self._init_weights()
        self.fc_weights_shape = self.weights.shape

    def _init_weights(self, init=True):
        if init:
            weights = np.random.uniform(-1,1, size=(*self.prev_ly_shape, *self.layer_shape))
        else:
            weights = np.zeros(shape=(*self.prev_ly_shape, *self.layer_shape))
        return(weights)


class Conv_Block:
    def __init__(self, num_filters, kernel_shape, stride, layer_shape, index):
        self.num_filters = num_filters
        self.kernel_shape = kernel_shape
        self.stride = stride
        self.layer_shape = layer_shape
        self.index = index

        self.b_type = "conv"
        self.ly_dim = len(layer_shape)
        self.filters = self.init_filters(self.num_filters, self.kernel_shape, init = True)
        self.feature_maps = self._init_feature_maps()
        self.layer, self.z_values = self._init_layer()
        self.biases = np.zeros(layer_shape) #idk if i need this but we can remove it later

    def _init_feature_maps(self, init=True):
        if init:
            feature_maps = np.random.uniform(-1,1, size=(self.num_filters, *self.layer_shape))
        else:
            feature_maps = np.zeros(shape=(self.num_filters, *self.layer_shape))
        return(feature_maps)

    def _init_layer(self):
        l_activations = np.zeros(self.layer_shape)
        z_values = np.zeros(self.layer_shape)
        return(l_activations, z_values)

    @staticmethod    
    def init_filters(num_filters:int, kernel_shape:tuple, init=True):
        if init:
            filters = np.random.uniform(-1,1, size=(*kernel_shape, num_filters))
        else:
            filters = np.zeros((*kernel_shape, num_filters))
        return (filters)

class NN:
    def __init__(self, config:DictConfig, blocks: list):
        self.config = config
        self.blocks = blocks

    @classmethod
    def create_network(cls, cfg:DictConfig, **kwargs):
        index_to_block = build_index_lookup(cfg)

        blocks = []
        for items in cfg.blocks:
            print(items)
            block_name, block_data = index_to_block[cfg.blocks[items].index]
            

            if block_data.type == "input":
                blocks.append(Input_block(block_data.shape, block_data.index))
            if block_data.type == "fc":
                blocks.append(FC_block(prev_bk_data.shape, block_data.shape, block_data.index))
            if block_data.type == "conv2D":
                fltr = block_data.filters
                blocks.append(Conv_Block(fltr.filter_num, fltr.kernel_shape, fltr.stride, block_data.shape, block_data.index))

            prev_bk_name = block_name
            prev_bk_data = block_data

        return cls(cfg, blocks)

    



def forward(input_vals:np.ndarray, net:NN):
    net.layers[0] = input_vals


    for index in range(1, len(net.layers), 1):
        shape = net.layers[index].activations.shape
        new_z_ly = np.zeros(shape)
        new_ly   = np.zeros(shape)
        #net.z_layers[index] = np.dot(net.layers[(index-1)], net.weights[index-1]) + net.biases[index-1]
        #todo: find a preforment way to do this!
        for i in shape[0]:
            for j in shape[1]:
                new_z_ly[i,j] = net.layers[(index - 1)].activations[i,j]* net.weights[index-1].weights[i,j]
                



#todo:
    #figure out network autocreation DONE!
    #figure out padding algorrithm DONE!
    #figure out down sizing 
        #this will be through pooling
    #write forward functions
        #since these differ between different types of layers and blocks, 
        # maybe each block should have a forward function?
        

In [12]:
cfg = OmegaConf.load("model.yaml")

model = NN.create_network(cfg)

input_ly
conv_block_1
conv_block_2
conv_block_3
output_ly


In [13]:
item = model.blocks
print("blocks")
for items in item:
    print(items.layer_shape, items.index, items.b_type)




blocks
[28, 28] 0 input
[28, 28] 1 conv
[14, 14] 2 conv
[14, 14] 3 conv
[10] 4 fc


In [ ]:
cfg = OmegaConf.load("model.yaml")
print(type(cfg))
print(cfg)
examine_1 = cfg.model.layers.input_ly.shape
print("\nexamine_1: ")
print(examine_1)
print(type(examine_1))
examine_2 = tuple(cfg.model.layers.input_ly.shape)
print("\nexamine_2: ")
print(examine_2)
print(type(examine_2))
#next to figure out how to handle .yaml files and DictConfig files

#net = NN.create_network(cfg)


In [ ]:
import numpy as np
filters  = {"prev_ly_size":28,
            "kernel_size": 3,
            "stride" : 1}
l1 = np.zeros(shape=(filters["prev_ly_size"],))
l11 = np.zeros(shape=(filters["prev_ly_size"],))

l2 = []
l1[0] = 1
for index, values in enumerate(l1):
    if index%filters["stride"] == 0:
        l1[index] = 1


    if l1[index] == 1:
        for x in range(filters["kernel_size"]): 
            y=x+1
            try:
                l11[index + (y - filters["kernel_size"]//2)] = l11[index + (y - filters["kernel_size"]//2)] + 1
            except IndexError as error:
                print("had an index error, continuing")
print(l1,"\n",l11)